# TaNP Gating-FiLM: три датасета на Colab T4
Ноутбук подготавливает данные и последовательно обучает отдельную модель для LastFM, MovieLens-1M и Gowalla. Все вычисления запускаются через `run_experiment.py`.

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/vladimirlavrentyev/cu-cold-start-recommendation-research.git'
PROJECT = Path('/content/cu-cold-start-recommendation-research')
if PROJECT.exists():
    !git -C {PROJECT} pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}

In [ ]:
import torch
assert torch.cuda.is_available(), 'В Colab выберите Runtime → Change runtime type → T4 GPU'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
!python -m pip install -q -r requirements-colab.txt

## Подготовка данных
LastFM извлекается из авторского архива. MovieLens-1M загружается с GroupLens. Gowalla строится детерминированно из NGCF; точный подготовленный сплит Gowalla авторы TaNP не опубликовали, поэтому это реконструкция протокола.

In [ ]:
!python scripts/prepare_datasets.py --dataset all --seed 2020 --gowalla-max-users 2692

In [ ]:
!python run_experiment.py --suite configs/tanp_gating_film_colab.json --dry-run

## Обучение
Следующая ячейка обучает три модели последовательно. При повторном запуске `--skip-existing` не переобучает уже завершённые датасеты.

In [ ]:
!python run_experiment.py --suite configs/tanp_gating_film_colab.json --skip-existing

## Сохранение результатов в Google Drive

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
archive = shutil.make_archive(
    '/content/drive/MyDrive/tanp_gating_film_seed2020',
    'zip',
    root_dir=PROJECT,
    base_dir='artifacts/runs',
)
print('Checkpoints:', archive)
shutil.copytree(PROJECT / 'results', '/content/drive/MyDrive/tanp_results', dirs_exist_ok=True)
print('Таблицы скопированы в MyDrive/tanp_results')